# LeetCode 748 — Shortest Completing Word bằng Python

**Link đề:** https://leetcode.com/problems/shortest-completing-word/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, String

```python
class Solution:
    def shortestCompletingWord(self, licensePlate: str, words: list[str]) -> str:
        ...
```

## Mục tiêu học
1. Đọc đúng đề: bỏ số/khoảng trắng, **không phân biệt hoa thường**, chữ lặp lại phải xuất hiện **đủ số lần**; chọn từ **ngắn nhất**, hòa thì lấy từ **xuất hiện trước**.
2. Ôn `str.isalpha`, `str.lower`, `str.count`, `collections.Counter` và phép trừ `Counter`.
3. Đi từ cách đếm thủ công → `Counter` (lời giải chính) → mảng 26 ô → mẹo sáng tạo **tích số nguyên tố**.
4. Trace, chứng minh, phân tích độ phức tạp, test đối chiếu.
5. Luyện các biến thể cùng kỹ thuật "**đếm tần suất và so sánh kho chữ**" (Ransom Note, Find Words That Can Be Formed...).

## 1. Đề bài

Cho chuỗi `licensePlate` (biển số xe) và mảng `words`.

Một **completing word** (từ hoàn chỉnh) là từ chứa **mọi chữ cái** của `licensePlate`, với quy tắc:
- **Bỏ qua** chữ số và dấu cách trong `licensePlate`.
- **Không phân biệt hoa thường** (`'P'` coi như `'p'`).
- Chữ cái xuất hiện **k lần** trong biển số thì phải xuất hiện **ít nhất k lần** trong từ.

Trả về completing word **ngắn nhất**. Nếu nhiều từ cùng ngắn nhất → lấy từ **xuất hiện đầu tiên** trong `words`. Đề bảo đảm luôn có đáp án.

Ví dụ minh họa trong đề: `licensePlate = "aBc 12c"` → cần `a`, `b`, `c`×2. Các từ hợp lệ: `"abccdef"`, `"caaacab"`, `"cbca"`.

| licensePlate | words | Output |
|---|---|:---:|
| `"1s3 PSt"` | `["step","steps","stripe","stepple"]` | `"steps"` |
| `"1s3 456"` | `["looks","pest","stew","show"]` | `"pest"` |

Ví dụ 1: biển số cần `s`×2, `p`, `t`. `"step"` chỉ có 1 `s` → loại; `"steps"` đủ → chọn. Ví dụ 2: chỉ cần `s`; `"pest"`, `"stew"`, `"show"` cùng dài 4 → lấy `"pest"` (đứng trước).

**Constraints:** `1 <= len(licensePlate) <= 7` (chữ số, chữ hoa/thường, dấu cách); `1 <= len(words) <= 1000`; `1 <= len(words[i]) <= 15`, chỉ chữ thường.

## 2. Nền tảng Python

### 2.1. Kiểm tra loại ký tự và đổi chữ thường

- `ch.isalpha()` → `True` nếu `ch` là chữ cái.
- `ch.isdigit()` → `True` nếu là chữ số.
- `s.lower()` → bản chữ thường của chuỗi.

In [1]:
plate = "1s3 PSt"
for ch in plate:
    print(repr(ch), "isalpha:", ch.isalpha(), "| isdigit:", ch.isdigit())
letters = [ch.lower() for ch in plate if ch.isalpha()]
print("Các chữ cái cần có:", letters)

'1' isalpha: False | isdigit: True
's' isalpha: True | isdigit: False
'3' isalpha: False | isdigit: True
' ' isalpha: False | isdigit: False
'P' isalpha: True | isdigit: False
'S' isalpha: True | isdigit: False
't' isalpha: True | isdigit: False
Các chữ cái cần có: ['s', 'p', 's', 't']


### 2.2. `str.count` và `''.join`

- `word.count(ch)` đếm số lần `ch` xuất hiện trong `word` (`O(len(word))`).
- `''.join(list_of_chars)` nối list ký tự thành chuỗi.

In [2]:
print("steps".count("s"), "step".count("s"), "stew".count("x"))
print("".join(["s", "p", "s", "t"]))

2 1 0
spst


### 2.3. `collections.Counter` — đếm tần suất

`Counter(iterable)` tạo dict đặc biệt: key = phần tử, value = số lần xuất hiện. Truy cập key không có → trả `0` (không lỗi).

In [3]:
from collections import Counter

need = Counter("spst")
print(need)                  # Counter({'s': 2, 'p': 1, 't': 1})
print(need["s"], need["z"])  # 2 0
print(Counter("steps"))

Counter({'s': 2, 'p': 1, 't': 1})
2 0
Counter({'s': 2, 't': 1, 'e': 1, 'p': 1})


### 2.4. Phép trừ `Counter` — "còn thiếu bao nhiêu?"

`A - B` trừ từng key và **chỉ giữ các giá trị dương**. Vì vậy `need - Counter(word)` chính là "**những chữ còn thiếu**" sau khi dùng các chữ của `word`. Rỗng (falsy) ⟺ `word` đủ chữ.

In [4]:
need = Counter("spst")
for w in ["step", "steps", "stripe"]:
    missing = need - Counter(w)
    print(f"{w:<7} thiếu: {dict(missing)}  -> {'đủ' if not missing else 'không đủ'}")

step    thiếu: {'s': 1}  -> không đủ
steps   thiếu: {}  -> đủ
stripe  thiếu: {'s': 1}  -> không đủ


`not missing` → `True` khi `missing` rỗng (một `Counter`/dict rỗng được coi là `False` trong ngữ cảnh điều kiện).

### 2.5. `<=` giữa hai `Counter` (Python 3.10+)

`need <= Counter(word)` kiểm "mọi số đếm trong `need` đều ≤ số đếm tương ứng trong `word`" — đúng nghĩa "**chứa đủ**". Cell dưới kiểm tra phiên bản để chạy an toàn.

In [5]:
import sys
print("Python", sys.version.split()[0])
if sys.version_info >= (3, 10):
    print(Counter("spst") <= Counter("steps"), Counter("spst") <= Counter("step"))
else:
    print("Phiên bản < 3.10: dùng phép trừ Counter thay thế")

Python 3.13.18
True False


## 3. Chia bài làm hai bước

1. **Chuẩn hóa biển số** → `need` (đếm các chữ cái thường).
2. **Duyệt `words` theo thứ tự**, với mỗi từ hỏi "có đủ chữ không?". Trong các từ đủ, giữ từ **ngắn nhất**; chỉ thay khi gặp từ **ngắn hơn hẳn** (`<`) — nhờ vậy khi hòa ta giữ từ đứng trước.

Phần thay đổi giữa các cách chỉ là: **kiểm tra "đủ chữ" thế nào**.

In [6]:
def normalize_plate(licensePlate: str) -> str:
    return "".join(ch.lower() for ch in licensePlate if ch.isalpha())

for p in ["1s3 PSt", "1s3 456", "aBc 12c", "GrC8950"]:
    print(f"{p!r:>10} -> {normalize_plate(p)!r}")

 '1s3 PSt' -> 'spst'
 '1s3 456' -> 's'
 'aBc 12c' -> 'abcc'
 'GrC8950' -> 'grc'


## 4. Cách 1 — Trực giác: dùng `str.count` cho từng chữ

Từ `word` đủ chữ ⟺ với mỗi chữ `ch` khác nhau trong biển số: `word.count(ch) >= plate.count(ch)`.

In [7]:
def covers_count(word: str, plate_letters: str) -> bool:
    for ch in set(plate_letters):
        if word.count(ch) < plate_letters.count(ch):
            return False
    return True


def shortest_count(licensePlate: str, words: list[str]) -> str:
    plate = normalize_plate(licensePlate)
    best = None
    for w in words:
        if covers_count(w, plate) and (best is None or len(w) < len(best)):
            best = w
    return best

print(shortest_count("1s3 PSt", ["step", "steps", "stripe", "stepple"]))
print(shortest_count("1s3 456", ["looks", "pest", "stew", "show"]))

steps
pest


Với `L ≤ 7` chữ cái trên biển số và từ dài `≤ 15`, cách này chạy nhanh: mỗi từ tốn khoảng `7 × 15` phép so sánh ký tự. Nhưng nó **đếm lại** từ đầu cho mỗi chữ — với biển số dài hơn sẽ thành `O(L · len(word))` mỗi từ.

`best is None or len(w) < len(best)`: nhờ **ngắn mạch** của `or`, khi `best` còn là `None` ta không gọi `len(None)` (sẽ lỗi).

## 5. Cách 2 — `Counter` (lời giải chính)

- Đếm biển số **một lần**: `need = Counter(normalize_plate(licensePlate))`.
- Mỗi từ: đếm một lượt `Counter(w)`, so với `need` bằng phép trừ.
- Mẹo nhỏ: nếu `len(w) >= len(best)` thì **không cần** kiểm tra (không thể tốt hơn) → bỏ qua sớm.

In [8]:
def shortest_counter(licensePlate: str, words: list[str]) -> str:
    need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
    best = None
    for w in words:
        if best is not None and len(w) >= len(best):
            continue                         # không thể ngắn hơn -> khỏi kiểm
        if not (need - Counter(w)):          # không thiếu chữ nào
            best = w
    return best

print(shortest_counter("1s3 PSt", ["step", "steps", "stripe", "stepple"]))
print(shortest_counter("1s3 456", ["looks", "pest", "stew", "show"]))

steps
pest


## 6. Trace từng bước

`licensePlate = "1s3 PSt"` → `need = {s: 2, p: 1, t: 1}`.

| word | len | best trước | Bỏ qua sớm? | Counter(word) | need − Counter(word) | best sau |
|---|---:|---|:---:|---|---|---|
| step | 4 | None | không | s1 t1 e1 p1 | {s: 1} | None |
| steps | 5 | None | không | s2 t1 e1 p1 | {} ✅ | steps |
| stripe | 6 | steps | **có** (6 ≥ 5) | — | — | steps |
| stepple | 7 | steps | **có** | — | — | steps |

In [9]:
def trace_shortest(licensePlate, words):
    need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
    print("need =", dict(need))
    best = None
    for w in words:
        if best is not None and len(w) >= len(best):
            print(f"{w:<8} len={len(w)} >= len(best)={len(best)} -> bỏ qua sớm")
            continue
        missing = need - Counter(w)
        if not missing:
            print(f"{w:<8} len={len(w)} đủ chữ -> best = {w!r}")
            best = w
        else:
            print(f"{w:<8} len={len(w)} thiếu {dict(missing)}")
    print("=> kết quả:", best)
    return best

trace_shortest("1s3 PSt", ["step", "steps", "stripe", "stepple"])
print()
trace_shortest("1s3 456", ["looks", "pest", "stew", "show"])

need = {'s': 2, 'p': 1, 't': 1}
step     len=4 thiếu {'s': 1}
steps    len=5 đủ chữ -> best = 'steps'
stripe   len=6 >= len(best)=5 -> bỏ qua sớm
stepple  len=7 >= len(best)=5 -> bỏ qua sớm
=> kết quả: steps

need = {'s': 1}
looks    len=5 đủ chữ -> best = 'looks'
pest     len=4 đủ chữ -> best = 'pest'
stew     len=4 >= len(best)=4 -> bỏ qua sớm
show     len=4 >= len(best)=4 -> bỏ qua sớm
=> kết quả: pest


'pest'

Ở ví dụ 2: `"looks"` đủ chữ → best; `"pest"` ngắn hơn (4 < 5) và đủ → best; `"stew"`, `"show"` có `4 >= 4` → bỏ qua — chính quy tắc "hòa thì giữ từ đứng trước" được thực thi bởi dấu `>=` này.

## 7. Cách 3 — Mảng 26 ô (không dùng thư viện)

Mỗi chữ `ch` ứng với ô `ord(ch) - ord('a')` (0..25). Đếm biển số vào `need[26]`, đếm từ vào `have[26]`, rồi so từng ô. Đây là cách làm "chuẩn" trong các ngôn ngữ không có `Counter`.

In [10]:
print({ch: ord(ch) - ord('a') for ch in "aesz"})

def count26(s: str) -> list[int]:
    cnt = [0] * 26
    for ch in s:
        cnt[ord(ch) - ord('a')] += 1
    return cnt


def shortest_array(licensePlate: str, words: list[str]) -> str:
    need = count26(normalize_plate(licensePlate))
    best = None
    for w in words:
        if best is not None and len(w) >= len(best):
            continue
        have = count26(w)
        if all(have[k] >= need[k] for k in range(26)):
            best = w
    return best

print(shortest_array("1s3 PSt", ["step", "steps", "stripe", "stepple"]))
print(shortest_array("1s3 456", ["looks", "pest", "stew", "show"]))

{'a': 0, 'e': 4, 's': 18, 'z': 25}
steps
pest


## 8. Cách 4 — Sáng tạo: tích số nguyên tố

Gán mỗi chữ một **số nguyên tố** khác nhau: `a→2, b→3, c→5, d→7, ...`. "Chữ ký" của một chuỗi = **tích** các số nguyên tố của các chữ trong nó.

Theo **định lý cơ bản của số học** (mỗi số phân tích thành thừa số nguyên tố **duy nhất**):

```text
word chứa đủ chữ của plate  ⟺  sig(plate) chia hết sig(word)
```

Ví dụ `plate = "spst"`: `sig = 67·53·67·71`. `"steps"` có `sig` chứa `67²·53·71·(e)` → chia hết ✔; `"step"` chỉ có `67¹` → không chia hết ✘.

Python có số nguyên **không giới hạn**, nên tích của 15 số nguyên tố (≤ 101) vẫn tính chính xác.

In [11]:
PRIMES = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47,
          53, 59, 61, 67, 71, 73, 79, 83, 89, 97, 101]
assert len(PRIMES) == 26

def signature(s: str) -> int:
    prod = 1
    for ch in s:
        prod *= PRIMES[ord(ch) - ord('a')]
    return prod

sp = signature("spst")
for w in ["step", "steps", "stripe"]:
    sw = signature(w)
    print(f"{w:<7} sig={sw:<14} sig % sig(plate) = {sw % sp:<10} -> {'đủ' if sw % sp == 0 else 'không đủ'}")

step    sig=2773331        sig % sig(plate) = 2773331    -> không đủ
steps   sig=185813177      sig % sig(plate) = 0          -> đủ
stripe  sig=3890983393     sig % sig(plate) = 5798783    -> không đủ


In [12]:
def shortest_primes(licensePlate: str, words: list[str]) -> str:
    target = signature(normalize_plate(licensePlate))
    best = None
    for w in words:
        if (best is None or len(w) < len(best)) and signature(w) % target == 0:
            best = w
    return best

print(shortest_primes("1s3 PSt", ["step", "steps", "stripe", "stepple"]))
print(shortest_primes("1s3 456", ["looks", "pest", "stew", "show"]))

steps
pest


Mẹo này đẹp về mặt toán học, nhưng phép nhân/chia số lớn chậm hơn đếm, và trong ngôn ngữ có số nguyên giới hạn (C++/Java) sẽ **tràn số**. Dùng để mở rộng tư duy; khi phỏng vấn nên trình bày `Counter`/mảng 26 trước.

## 9. Code nộp LeetCode

### Bản `Counter` (khuyến nghị)

In [13]:
from collections import Counter


class Solution:
    def shortestCompletingWord(self, licensePlate: str, words: list[str]) -> str:
        need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
        best = None
        for w in words:
            if best is not None and len(w) >= len(best):
                continue
            if not (need - Counter(w)):
                best = w
        return best


sol = Solution()
print(sol.shortestCompletingWord("1s3 PSt", ["step", "steps", "stripe", "stepple"]))
print(sol.shortestCompletingWord("1s3 456", ["looks", "pest", "stew", "show"]))

steps
pest


### Bản mảng 26 ô và bản `min` một dòng

`min(iterable, key=len)` trả phần tử có `len` nhỏ nhất; khi hòa, `min` trả phần tử **xuất hiện đầu tiên** — đúng quy tắc của đề. Trên LeetCode đổi tên class thành `Solution`.

In [14]:
class SolutionArray:
    def shortestCompletingWord(self, licensePlate: str, words: list[str]) -> str:
        need = [0] * 26
        for ch in licensePlate:
            if ch.isalpha():
                need[ord(ch.lower()) - 97] += 1
        best = None
        for w in words:
            if best is not None and len(w) >= len(best):
                continue
            have = [0] * 26
            for ch in w:
                have[ord(ch) - 97] += 1
            if all(h >= n for h, n in zip(have, need)):
                best = w
        return best


class SolutionMin:
    def shortestCompletingWord(self, licensePlate: str, words: list[str]) -> str:
        need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
        return min((w for w in words if not (need - Counter(w))), key=len)


print(SolutionArray().shortestCompletingWord("1s3 456", ["looks", "pest", "stew", "show"]),
      SolutionMin().shortestCompletingWord("1s3 456", ["looks", "pest", "stew", "show"]))

pest pest


`97` là `ord('a')`. `zip(have, need)` ghép từng ô của hai mảng 26 phần tử.

## 10. Vì sao đúng?

**Kiểm tra "đủ chữ".** `need - Counter(w)` chứa key `ch` với giá trị `need[ch] - cnt_w[ch]` **nếu dương**. Rỗng ⟺ với mọi `ch`, `cnt_w[ch] >= need[ch]` ⟺ đúng định nghĩa completing word (sau khi đã bỏ số, dấu cách, đổi chữ thường).

**Chọn đúng từ.** Invariant: sau khi xét `words[0..j]`, `best` là completing word **ngắn nhất, xuất hiện sớm nhất** trong đoạn đó (hoặc `None`).
- Gặp `w` với `len(w) >= len(best)`: `w` không thể thay `best` (dài hơn, hoặc bằng nhưng đứng sau) → bỏ qua đúng.
- Gặp `w` ngắn hơn hẳn và đủ chữ → `w` tốt hơn mọi từ trước → cập nhật đúng.
- Ngắn hơn nhưng không đủ chữ → không phải ứng viên.

Đề bảo đảm có đáp án nên cuối cùng `best` khác `None`.

**Cách 4.** Phân tích thừa số nguyên tố duy nhất ⇒ số mũ của prime `p_ch` trong `sig(s)` bằng số lần `ch` xuất hiện trong `s`. `sig(plate) | sig(w)` ⟺ mọi số mũ trong plate ≤ số mũ tương ứng trong `w` ⟺ đủ chữ.

## 11. Độ phức tạp

Gọi `P = len(licensePlate)` (≤ 7), `W = len(words)` (≤ 1000), `K = len(words[i])` tối đa (≤ 15), `Σ = 26` (kích thước bảng chữ cái).

| Cách | Time | Space phụ |
|---|---|---|
| 1. `str.count` | `O(W · P · K)` | `O(P)` |
| 2. `Counter` | `O(P + W · K)` (phép trừ tốn `O(số key của need) ≤ O(P)`) | `O(Σ)` |
| 3. Mảng 26 | `O(P + W · (K + Σ))` | `O(Σ)` |
| 4. Tích nguyên tố | `O(P + W · K)` phép nhân số lớn | `O(1)` số nguyên (nhưng số có thể lớn) |

Tổng ký tự của `words` tối đa `15 000` → mọi cách đều rất nhanh. Bước "bỏ qua sớm khi `len(w) >= len(best)`" không đổi Big-O nhưng giảm việc thực tế.

## 12. Bộ test

In [15]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.shortestCompletingWord("1s3 PSt", ["step", "steps", "stripe", "stepple"]), "steps")
check("đề 2 (hòa -> đứng trước)", sol.shortestCompletingWord("1s3 456", ["looks", "pest", "stew", "show"]), "pest")
check("ví dụ aBc 12c", sol.shortestCompletingWord("aBc 12c", ["abccdef", "caaacab", "cbca"]), "cbca")
check("hoa thường", sol.shortestCompletingWord("ABC", ["cba", "ab"]), "cba")
check("chữ lặp", sol.shortestCompletingWord("Ss", ["s", "sus", "ss"]), "ss")
check("từ đầu tiên đã tốt nhất", sol.shortestCompletingWord("a", ["a", "b", "aa"]), "a")
check("từ tốt nhất ở cuối", sol.shortestCompletingWord("t 1", ["tttt", "ttt", "tt", "t"]), "t")
check("biển số GrC8950", sol.shortestCompletingWord("GrC8950", ["measure", "other", "every", "base", "according", "level", "meeting", "none", "marriage", "rest"]), "according")

PASS  đề 1: 'steps'
PASS  đề 2 (hòa -> đứng trước): 'pest'
PASS  ví dụ aBc 12c: 'cbca'
PASS  hoa thường: 'cba'
PASS  chữ lặp: 'ss'
PASS  từ đầu tiên đã tốt nhất: 'a'
PASS  từ tốt nhất ở cuối: 't'
PASS  biển số GrC8950: 'according'


In [16]:
import random
import string

rng = random.Random(748)
impls = [shortest_count, shortest_counter, shortest_array, shortest_primes,
         Solution().shortestCompletingWord, SolutionArray().shortestCompletingWord, SolutionMin().shortestCompletingWord]
alpha = "abcde"
for _ in range(3000):
    plate_letters = [rng.choice(alpha + alpha.upper()) for _ in range(rng.randint(1, 4))]
    plate_chars = plate_letters + [rng.choice("0123456789 ") for _ in range(rng.randint(0, 3))]
    rng.shuffle(plate_chars)
    plate = "".join(plate_chars)
    words = ["".join(rng.choice(alpha) for _ in range(rng.randint(1, 8))) for _ in range(rng.randint(1, 8))]
    words.append(normalize_plate(plate) + "".join(rng.choice(alpha) for _ in range(rng.randint(0, 3))))  # bảo đảm có đáp án
    rng.shuffle(words)
    outs = {f(plate, words) for f in impls}
    assert len(outs) == 1, (plate, words, outs)
print("3000 bộ ngẫu nhiên: 7 cài đặt khớp nhau")

3000 bộ ngẫu nhiên: 7 cài đặt khớp nhau


Bộ sinh dữ liệu luôn chèn một từ chứa đủ chữ của biển số → đúng constraint "luôn có đáp án". `rng.shuffle` xáo trộn list tại chỗ để biển số có chữ/số/dấu cách lẫn lộn.

## 13. Lỗi người mới hay gặp

### Lỗi 1: Dùng `set` — mất thông tin chữ lặp

In [17]:
def wrong_set(licensePlate, words):
    need = {ch.lower() for ch in licensePlate if ch.isalpha()}
    best = None
    for w in words:
        if need <= set(w) and (best is None or len(w) < len(best)):
            best = w
    return best

args = ("1s3 PSt", ["step", "steps", "stripe", "stepple"])
print("sai :", wrong_set(*args), "| đúng:", shortest_counter(*args))

sai : step | đúng: steps


`"step"` có tập chữ `{s,t,e,p}` ⊇ `{s,p,t}` nhưng chỉ có **một** `s`. Phải đếm số lần → `Counter`.

### Lỗi 2: Quên đổi chữ thường

In [18]:
def wrong_case(licensePlate, words):
    need = Counter(ch for ch in licensePlate if ch.isalpha())     # thiếu .lower()
    best = None
    for w in words:
        if not (need - Counter(w)) and (best is None or len(w) < len(best)):
            best = w
    return best

print("sai :", wrong_case(*args), "| đúng:", shortest_counter(*args))

sai : None | đúng: steps


`'P'`, `'S'` (hoa) không bao giờ khớp chữ thường trong `words` → không từ nào đủ → `None`.

### Lỗi 3: Dùng `<=` khi so độ dài → chọn từ hòa **đứng sau**

In [19]:
def wrong_tie(licensePlate, words):
    need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
    best = None
    for w in words:
        if not (need - Counter(w)) and (best is None or len(w) <= len(best)):   # SAI: <=
            best = w
    return best

args2 = ("1s3 456", ["looks", "pest", "stew", "show"])
print("sai :", wrong_tie(*args2), "| đúng:", shortest_counter(*args2))

sai : show | đúng: pest


### Lỗi 4: Đếm cả chữ số / dấu cách

`Counter("1s3 PSt".lower())` có cả `'1'`, `'3'`, `' '` → không từ nào chứa → sai. Luôn lọc `isalpha()`.

In [20]:
print(Counter("1s3 PSt".lower()))
print(Counter(ch.lower() for ch in "1s3 PSt" if ch.isalpha()))

Counter({'s': 2, '1': 1, '3': 1, ' ': 1, 'p': 1, 't': 1})
Counter({'s': 2, 'p': 1, 't': 1})


### Lỗi 5: Sắp xếp `words` theo độ dài rồi lấy từ đầu tiên đủ — nhưng sắp bằng thứ tự **không ổn định** hoặc theo cả chữ cái

`sorted(words, key=len)` của Python là **ổn định** (stable): các từ cùng độ dài giữ thứ tự gốc → dùng được. Nhưng `sorted(words)` (theo chữ cái) hay sắp theo `(len, w)` sẽ phá thứ tự gốc khi hòa.

In [21]:
words = ["looks", "stew", "pest", "show"]
need = Counter("s")
print("sorted key=len    :", next(w for w in sorted(words, key=len) if not (need - Counter(w))))
print("sorted (len, w) SAI:", next(w for w in sorted(words, key=lambda w: (len(w), w)) if not (need - Counter(w))))
print("đúng               :", shortest_counter("1s3 456", words))

sorted key=len    : stew
sorted (len, w) SAI: pest
đúng               : stew


## 14. Mẹo nhận dạng pattern

- "Chuỗi A có **đủ** chữ để tạo/chứa chuỗi B không" (tính cả số lần lặp) → **đếm tần suất** (`Counter` hoặc mảng 26) rồi so từng chữ.
- `Counter`: `need - have` rỗng ⟺ đủ; `need <= have` (3.10+).
- "Không phân biệt hoa thường", "bỏ ký tự không phải chữ" → **chuẩn hóa** input trước, tách thành hàm riêng.
- "Ngắn nhất, hòa thì lấy cái đầu" → duyệt theo thứ tự và chỉ cập nhật khi **tốt hơn hẳn** (`<`), hoặc `min(..., key=len)`.
- Bài cùng họ: 383 Ransom Note, 1160 Find Words That Can Be Formed by Characters, 242 Valid Anagram, 438 Find All Anagrams, 76 Minimum Window Substring (Hard).

## 15. Biến thể

### 15.1. LeetCode 383 — Ransom Note

`ransomNote` có tạo được từ các chữ của `magazine` không (mỗi chữ dùng một lần)? Chính là "đủ chữ".

In [22]:
def can_construct(ransomNote: str, magazine: str) -> bool:
    return not (Counter(ransomNote) - Counter(magazine))

print(can_construct("a", "b"), can_construct("aa", "ab"), can_construct("aa", "aab"))   # False False True

False False True


### 15.2. LeetCode 1160 — Find Words That Can Be Formed by Characters

Tính tổng độ dài các từ "tạo được" từ kho chữ `chars` — đảo vai: giờ **từ** phải nằm gọn trong **kho**.

In [23]:
def count_characters(words, chars):
    pool = Counter(chars)
    return sum(len(w) for w in words if not (Counter(w) - pool))

print(count_characters(["cat", "bt", "hat", "tree"], "atach"))          # 6
print(count_characters(["hello", "world", "leetcode"], "welldonehoneyr"))  # 10

6
10


### 15.3. Trả về **tất cả** completing word ngắn nhất

Thay vì chỉ một, trả list mọi từ hợp lệ có độ dài nhỏ nhất (giữ thứ tự gốc).

In [24]:
def all_shortest(licensePlate, words):
    need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
    ok = [w for w in words if not (need - Counter(w))]
    m = min(map(len, ok))
    return [w for w in ok if len(w) == m]

print(all_shortest("1s3 456", ["looks", "pest", "stew", "show"]))

['pest', 'stew', 'show']


`map(len, ok)` áp `len` lên từng phần tử của `ok` (tạo iterator độ dài).

## 16. Bài tập

### Bài 1 — Tính tay
`licensePlate = "Ah71752"`, `words = ["suggest","letter","of","husband","easy","education","drug","prevent","writer","old"]`. Cần chữ gì? Đáp án là từ nào?

### Bài 2 — Chuẩn hóa biển số
Viết `plate_counter(licensePlate)` trả `Counter` các chữ cái thường.

In [25]:
def plate_counter(licensePlate: str) -> Counter:
    # TODO
    pass

# assert plate_counter("1s3 PSt") == Counter({"s": 2, "p": 1, "t": 1})
# assert plate_counter("123 456") == Counter()

### Bài 3 — Điền chỗ trống lời giải chính

In [26]:
def exercise_shortest(licensePlate: str, words: list[str]) -> str:
    need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())
    best = None
    for w in words:
        # TODO 1: bỏ qua nếu w không thể ngắn hơn best
        # TODO 2: nếu w đủ chữ -> best = w
        pass
    return best

# assert exercise_shortest("1s3 PSt", ["step", "steps", "stripe", "stepple"]) == "steps"
# assert exercise_shortest("1s3 456", ["looks", "pest", "stew", "show"]) == "pest"

### Bài 4 — Kiểm "đủ chữ" bằng mảng 26 ô
Viết `covers(word, plate_letters)` **không dùng `Counter`**, dùng mảng 26 ô và **một** mảng duy nhất (tăng cho word, giảm cho plate).

In [27]:
def covers(word: str, plate_letters: str) -> bool:
    # TODO
    pass

# assert covers("steps", "spst") == True
# assert covers("step", "spst") == False
# assert covers("abc", "") == True

### Bài 5 — Từ dài nhất tạo được từ kho chữ
Cho `pool` (chuỗi kho chữ) và `words`. Trả từ **dài nhất** tạo được từ `pool` (mỗi chữ dùng một lần); hòa thì lấy từ đứng trước; không có thì `""`.

In [28]:
def longest_formable(pool: str, words: list[str]) -> str:
    # TODO
    pass

# assert longest_formable("atach", ["cat", "bt", "hat", "tree"]) == "cat"
# assert longest_formable("xyz", ["a", "b"]) == ""

### Bài 6 — Mini challenge: chữ ký nguyên tố cho anagram
Dùng `signature` (tích nguyên tố) để **nhóm các anagram** (từ có cùng tập chữ với cùng số lần): trả list các nhóm theo thứ tự xuất hiện của từ đầu nhóm.

In [29]:
def group_anagrams_prime(words: list[str]) -> list[list[str]]:
    # TODO
    pass

# assert group_anagrams_prime(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]

## 17. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Bỏ số, đổi chữ thường | `need = {a: 1, h: 1}` | Từ ngắn nhất chứa cả `a` và `h` |
| 2 | Lọc + đổi chữ thường + đếm | Generator trong `Counter(...)` | `Counter(ch.lower() for ch in s if ch.isalpha())` |
| 3 | Như Cách 2 | `continue` khi `len(w) >= len(best)` | `if not (need - Counter(w)): best = w` |
| 4 | Một mảng: `+1` cho word, `-1` cho plate | Ô âm nghĩa là thiếu | `return all(v >= 0 for v in cnt)` |
| 5 | Đảo vai: từ ⊆ kho | Cập nhật khi **dài hơn hẳn** | `if not (Counter(w) - pool) and len(w) > len(best)` |
| 6 | Cùng chữ ký ⟺ anagram | `dict` chữ ký → list từ | `groups.setdefault(signature(w), []).append(w)`; dict giữ thứ tự chèn |

## 18. Đáp án tham khảo

**Bài 1:** cần `a` và `h`. Duyệt: `"suggest"`(không a), `"letter"`, `"of"` (không), `"husband"` (có `h`, `a` → độ dài 7, best), `"easy"` (không h), `"education"` (không h), `"drug"`, `"prevent"`, `"writer"`, `"old"` (không). Đáp án `"husband"`.

In [30]:
print("Bài 1:", trace_shortest("Ah71752", ["suggest", "letter", "of", "husband", "easy", "education", "drug", "prevent", "writer", "old"]))


def answer_plate_counter(licensePlate):
    return Counter(ch.lower() for ch in licensePlate if ch.isalpha())


def answer_exercise_shortest(licensePlate, words):
    need = answer_plate_counter(licensePlate)
    best = None
    for w in words:
        if best is not None and len(w) >= len(best):
            continue
        if not (need - Counter(w)):
            best = w
    return best


def answer_covers(word, plate_letters):
    cnt = [0] * 26
    for ch in word:
        cnt[ord(ch) - 97] += 1
    for ch in plate_letters:
        cnt[ord(ch) - 97] -= 1
    return all(v >= 0 for v in cnt)


def answer_longest_formable(pool, words):
    have = Counter(pool)
    best = ""
    for w in words:
        if len(w) > len(best) and not (Counter(w) - have):
            best = w
    return best


def answer_group_anagrams_prime(words):
    groups = {}
    for w in words:
        groups.setdefault(signature(w), []).append(w)
    return list(groups.values())


assert answer_plate_counter("1s3 PSt") == Counter({"s": 2, "p": 1, "t": 1})
assert answer_plate_counter("123 456") == Counter()
assert answer_exercise_shortest("1s3 PSt", ["step", "steps", "stripe", "stepple"]) == "steps"
assert answer_exercise_shortest("1s3 456", ["looks", "pest", "stew", "show"]) == "pest"
assert answer_covers("steps", "spst") is True
assert answer_covers("step", "spst") is False
assert answer_covers("abc", "") is True
assert answer_longest_formable("atach", ["cat", "bt", "hat", "tree"]) == "cat"
assert answer_longest_formable("xyz", ["a", "b"]) == ""
assert answer_group_anagrams_prime(["eat", "tea", "tan", "ate", "nat", "bat"]) == [["eat", "tea", "ate"], ["tan", "nat"], ["bat"]]
print("Tất cả đáp án PASS")

need = {'a': 1, 'h': 1}
suggest  len=7 thiếu {'a': 1, 'h': 1}
letter   len=6 thiếu {'a': 1, 'h': 1}
of       len=2 thiếu {'a': 1, 'h': 1}
husband  len=7 đủ chữ -> best = 'husband'
easy     len=4 thiếu {'h': 1}
education len=9 >= len(best)=7 -> bỏ qua sớm
drug     len=4 thiếu {'a': 1, 'h': 1}
prevent  len=7 >= len(best)=7 -> bỏ qua sớm
writer   len=6 thiếu {'a': 1, 'h': 1}
old      len=3 thiếu {'a': 1, 'h': 1}
=> kết quả: husband
Bài 1: husband
Tất cả đáp án PASS


Ghi chú bài 6: từ Python 3.7, `dict` **giữ thứ tự chèn**, nên `groups.values()` ra đúng thứ tự xuất hiện của từ đầu mỗi nhóm. `setdefault(key, [])` trả list sẵn có hoặc tạo list rỗng mới.

### Template nhớ nhanh + Checklist submit

```python
from collections import Counter

class Solution:
    def shortestCompletingWord(self, licensePlate, words):
        need = Counter(ch.lower() for ch in licensePlate if ch.isalpha())   # lọc + chữ thường + đếm
        best = None
        for w in words:
            if best is not None and len(w) >= len(best):   # >= : hòa giữ từ đứng trước
                continue
            if not (need - Counter(w)):                    # không thiếu chữ nào
                best = w
        return best
```

**Câu thần chú:** *"lọc chữ — hạ chữ thường — đếm bằng Counter — need trừ have rỗng là đủ — chỉ thay khi ngắn hơn hẳn."*

**Checklist trước khi submit:**
- [ ] Chỉ lấy `isalpha()` và `.lower()` từ biển số?
- [ ] Đếm **số lần** (Counter / mảng 26), không dùng `set`?
- [ ] So độ dài bằng `<` khi cập nhật (hoặc `>=` khi bỏ qua) để hòa giữ từ đứng trước?
- [ ] Không sắp xếp `words` làm mất thứ tự gốc (nếu sắp, chỉ `sorted(words, key=len)` ổn định)?
- [ ] Đã test chữ lặp (`"Ss"`), chữ hoa, và trường hợp hòa độ dài?